# Job-AI pipeline

End to end in one notebook:

1. **Résumé** → chunk `resume.yaml` → embed on the GPU
2. **Jobs** → scrape a board → LLM-revise + label → export eval
3. **Score** → rank every job against the résumé → `out/ranked_jobs.csv`

All logic lives in `src/` (`resume.py`, `pipeline.py`, `score.py`); this
notebook just drives it. Run the cells top to bottom.


## Setup: make the project's `src/` importable


In [1]:
import sys
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

from resume import (
    OUTPUT_PATH as RESUME_BULLETS,
    RESUME_PATH,
    flatten,
    load_or_embed,
    load_resume,
    save_chunks,
)
from pipeline import (
    DATA_DIR,
    LLMSettings,
    ScrapeConfig,
    make_client,
    revise,
    save_eval,
    save_revised,
    scrape,
    to_eval,
)
from score import OUT_CSV, aligned_resume_bullets, load_jobs, rank, write_csv

print(f"Project root: {ROOT}")

Project root: /home/yokesh/Projects/job-ai


## 1 · Résumé → chunks → embeddings

Validate `resume.yaml`, flatten it into `data/resume_bullets.json`, and embed
every chunk on the GPU (cached — instant re-runs until `resume.yaml` changes).


In [2]:
resume = load_resume(RESUME_PATH)
chunks = flatten(resume)
save_chunks(chunks, RESUME_BULLETS)

ids, resume_vecs = load_or_embed()
resume_bullets = aligned_resume_bullets(ids)
print(f"{len(chunks)} chunks, {len(ids)} embeddings {resume_vecs.shape}")

Cache hit — 26 embeddings from cache/resume_embeddings.npz
26 chunks, 26 embeddings (26, 1024)


## 2 · Job search settings

Edit the job title and board, then run this cell.


In [3]:
JOB_TITLE = "AI Engineering Manager"   # the search term
BOARD = "linkedin"                      # one of: linkedin indeed glassdoor google
LOCATION = "Canada"
RESULTS_WANTED = 10                     # how many postings to pull
HOURS_OLD = 168                         # only postings newer than this many hours (168 = 1 week)
IS_REMOTE = False

config = ScrapeConfig(
    site_name=[BOARD],
    search_term=JOB_TITLE,
    location=LOCATION,
    results_wanted=RESULTS_WANTED,
    hours_old=HOURS_OLD,
    is_remote=IS_REMOTE,
    country_indeed="canada",            # only used by indeed/glassdoor
    linkedin_fetch_description=True,    # pull the full JD text (needed to score)
)
config

ScrapeConfig(site_name=[<Site.linkedin: 'linkedin'>], search_term='AI Engineering Manager', location='Canada', google_search_term=None, results_wanted=10, distance=50, job_type=None, is_remote=False, hours_old=168, country_indeed='canada', linkedin_fetch_description=True, verbose=1)

## 3 · Scrape

Calls the board live (needs internet).


In [4]:
jobs = scrape(config)
print(f"Scraped {len(jobs)} jobs")

pd.DataFrame(
    {
        "title": [j.title for j in jobs],
        "company": [j.company for j in jobs],
        "has_description": [bool(j.description) for j in jobs],
    }
)

2026-06-20 16:32:06,314 - INFO - JobSpy:Linkedin - finished scraping


Scraped 10 jobs


,title,company,has_description
0,"Engineering Manager, Materia AI",Thomson Reuters,True
1,Lead AI Forward Engineer,Thomson Reuters,True
2,Generative AI Engineer -Vice President,Citi,True
3,"Manager, Software Development Engineering - AI...",Workday,True
4,"Manager of Technical Staff, Sovereign AI",Cohere,True
5,"Manager, Applied AI/ML, Data Science & Enginee...",Autodesk,True
6,"Engineering Manager, AI DocGen",EvenUp,True
7,"Senior Manager, AI Agents and Platform",Jerry,True
8,Engineering Manager - Machine Learning,Recursion,True
9,AI Engineering Lead,Definity,True


## 4 · Revise + export eval (LLM)

The LLM marks section boundaries and labels each JD sentence (`requirement`,
`responsibility`, `preferred`, `context`) — the text is never reworded. Writes
`jobs_<ts>_revised.jsonl` and `jobs_<ts>_eval.jsonl`. Makes live API calls.


In [5]:
settings = LLMSettings()  # reads OPENROUTER_* keys from .env
client = make_client(settings)
print(f"Using model: {settings.model}")

revised = revise(jobs, client, settings.model)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
revised_path = save_revised(revised, DATA_DIR / f"jobs_{stamp}.jsonl")
eval_path = save_eval([to_eval(job) for job in revised], revised_path)
print(f"Eval file -> {eval_path.relative_to(ROOT)}")

Using model: anthropic/claude-haiku-4.5
  batch 1/1 (10 jobs) — 2 LLM calls
    Engineering Manager, Materia AI — 4/10 sections kept, 33 chunks
    Lead AI Forward Engineer — 6/13 sections kept, 39 chunks
    Generative AI Engineer -Vice President — 4/10 sections kept, 27 chunks
    Manager, Software Development Engineering - AI Platform — 4/10 sections kept, 31 chunks
    Manager of Technical Staff, Sovereign AI — 3/7 sections kept, 20 chunks
    Manager, Applied AI/ML, Data Science & Engineering — 8/13 sections kept, 90 chunks
    Engineering Manager, AI DocGen — 4/8 sections kept, 35 chunks
    Senior Manager, AI Agents and Platform — 5/8 sections kept, 37 chunks
    Engineering Manager - Machine Learning — 3/8 sections kept, 25 chunks
    AI Engineering Lead — 5/10 sections kept, 38 chunks
Eval file -> data/jobs_20260620_163245_eval.jsonl


## 5 · Score & rank

Load the eval file we just wrote, embed each JD on the GPU (cached per job),
score against the résumé, and write the ranked CSV.


In [6]:
job_records = load_jobs(eval_path)
rows = rank(job_records, resume_vecs, resume_bullets)
write_csv(rows, OUT_CSV)
print(f"Wrote {len(rows)} ranked jobs -> {OUT_CSV.relative_to(ROOT)}")

Loaded: 10 jobs
Skipped: 0
Labels seen: responsibility=136, requirement=108, preferred=22, context=109


ranking:   0%|          | 0/10 [00:00<?, ?job/s]/home/yokesh/Projects/job-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1646.46it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded BAAI/bge-large-en-v1.5 on NVIDIA GeForce RTX 3050


ranking: 100%|██████████| 10/10 [00:14<00:00,  1.44s/job]

Embedded 10 job(s), 0 from cache
Wrote 10 ranked jobs -> out/ranked_jobs.csv


## Final ranked CSV

Composite score plus the per-label breakdown, best first.


In [7]:
pd.DataFrame(rows)[
    ["rank", "composite", "company", "position",
     "requirement", "responsibility", "preferred", "context"]
].head(15)

,rank,composite,company,position,requirement,responsibility,preferred,context
0,1,0.7089,Workday,"Manager, Software Development Engineering - AI...",0.7221,0.6911,0.7263,0.6306
1,2,0.6954,EvenUp,"Engineering Manager, AI DocGen",0.7372,0.6741,0.6438,0.5595
2,3,0.6680,Thomson Reuters,Lead AI Forward Engineer,0.6966,0.6406,0.6403,0.6289
3,4,0.6555,Thomson Reuters,"Engineering Manager, Materia AI",0.6825,0.6413,0.6286,0.5513
4,5,0.6437,Autodesk,"Manager, Applied AI/ML, Data Science & Enginee...",0.6426,0.6549,,0.5877
5,6,0.6276,Cohere,"Manager of Technical Staff, Sovereign AI",0.6310,0.6362,,0.5420
6,7,0.6118,Definity,AI Engineering Lead,0.6395,0.5954,0.5777,0.5365
7,8,0.6085,Jerry,"Senior Manager, AI Agents and Platform",0.5947,0.6442,,0.5318
8,9,0.6061,Citi,Generative AI Engineer -Vice President,0.6328,0.5976,0.5379,0.5954
9,10,0.5938,Recursion,Engineering Manager - Machine Learning,0.6334,0.6048,0.4765,0.4838
